In [ ]:
import numpy as np
import pandas as pd
from alibi_detect.cd import ChiSquareDrift

URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"
CAT_COLS = ["sex", "pclass", "embarked", "who", "alone"]

In [ ]:
# 1) Load Titanic and keep only categorical columns (+ survived for creating drift)
df = pd.read_csv(URL, usecols=CAT_COLS + ["survived"]).dropna(subset=CAT_COLS)

In [ ]:
# 2) Integer-encode with ONE shared mapping for all splits (ChiSquareDrift expects 0..n-1)
mappings = {}
for col in CAT_COLS:
    cat = pd.Categorical(df[col])
    df[col] = cat.codes.astype(np.int64)
    mappings[col] = dict(enumerate(cat.categories))

In [ ]:
# 3) Reference / test split
ref = df.sample(frac=0.5, random_state=42)
test = df.drop(ref.index)

x_train = ref[CAT_COLS].reset_index(drop=True)                    # pandas DataFrame
x_test_no_drift = test[CAT_COLS].reset_index(drop=True)           # same distribution
x_test_drift = test.loc[test["survived"] == 1, CAT_COLS].reset_index(drop=True)  # survivors only

In [ ]:
# 4) Detector: number of categories per feature index
categories_per_feature = {i: int(df[c].nunique()) for i, c in enumerate(CAT_COLS)}
cd = ChiSquareDrift(
    x_train.to_numpy(),
    p_val=0.05,
    categories_per_feature=categories_per_feature,
)

In [ ]:
# 5) Predict
for name, x in [("no drift", x_test_no_drift), ("drift", x_test_drift)]:
    batch = cd.predict(x.to_numpy(), drift_type="batch")
    feat = cd.predict(x.to_numpy(), drift_type="feature")
    print(f"\n=== {name} (n={len(x)}) ===")
    print("batch is_drift:", batch["data"]["is_drift"],
          "| Bonferroni threshold:", round(batch["data"]["threshold"], 4))
    print(pd.DataFrame({
        "feature": CAT_COLS,
        "chi2": np.round(feat["data"]["distance"], 2),
        "p_val": np.round(feat["data"]["p_val"], 4),
        "is_drift": feat["data"]["is_drift"],
    }).to_string(index=False))

print("\nEncodings:", mappings)